<a href="https://colab.research.google.com/github/zmhibner-gif/ml_internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zmhibner-gif/ml_internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

safe_token = HF_TOKEN.replace("'", "''")

con.execute(
    f"CREATE OR REPLACE SECRET hf_secret "
    f"(TYPE huggingface, TOKEN '{safe_token}')"
)

# Main warehouse path
WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"

print("Hugging Face connection ready.")

Hugging Face connection ready.


## 1. Unit of analysis + time window

**Unit of analysis:** One anonymized content page for one client, aggregated over one month. The warehouse contains daily observations, so I will combine the March rows into one monthly row per page.

**Time window:** March 2026. I am using a mid-panel month for development and keeping the final month separate for later testing.


In [ ]:
# Use March 2026 as the development month
MONTH = "2026-03"

WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{WAREHOUSE}/fact_content_daily_performance"
MAR = f"read_parquet('{FACT}/month={MONTH}/*.parquet')"

con.sql(f"""
    SELECT
        report_date,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position
    FROM {MAR}
    LIMIT 5
""").df()

,report_date,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position
0,2026-03-01,content_b7e512995f79d5a6,20,0,3.350000
1,2026-03-01,content_05597932fe4da067,1,0,0.000000
2,2026-03-01,content_7a105f548d9c6916,125,1,4.928000
3,2026-03-01,content_905aa32a0230694e,7,0,4.000000
4,2026-03-01,content_a3ea9792f793ec72,11,0,2.272727


## 2. Fields: feature / label / context / excluded

**Table used:** `fact_content_daily_performance`

**Features:**

* `impressions` — total search visibility during March
* `ctr` — percentage of impressions that resulted in clicks
* `avg_position` — average search position
* `days_with_impressions` — number of days in March when the page appeared in search results
* `days_with_clicks` — number of days in March when the page received at least one search click

**Label / proxy:** None. This is a clustering task, so there is no target label to predict. The eventual output will be a cluster assignment based on similarities between pages.

**Context:**

* `content_hash_id` — identifies the anonymized page
* `client_hash_id` — identifies the anonymized client
* `report_date` — identifies when the daily observation was recorded
* `gsc_data_available` and `ga4_data_available` — used to check data availability before choosing the final feature slice

**Excluded:**

* `content_hash_id` and `client_hash_id` are excluded from the clustering features because they are identifiers rather than measures of page performance.
* Fields that are not available or suitable for the final feature frame will be excluded after the availability checks in Section 3.




## 3. Verify it with queries (grain, counts, missing values, windows)



In [ ]:
# query 1 - grain
# Check that the raw warehouse has only one row per page, client, and day
grain_check = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS n
    FROM {MAR}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print("Duplicate rows found:", len(grain_check))
grain_check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate rows found: 0


,report_date,client_hash_id,content_hash_id,n


In [ ]:
# query 2 - row count and date span

slice_check = con.sql(f"""
    SELECT
        COUNT(*) AS rows,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM {MAR}
""").df()

slice_check

,rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [ ]:
# query 3 - availability
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS all_rows,

        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available,

        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
              AND gsc_impressions IS NOT NULL
              AND gsc_clicks IS NOT NULL
              AND gsc_sum_position IS NOT NULL
        ) AS gsc_feature_fields_complete,

        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_available,

        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
              AND ga4_pageviews IS NOT NULL
              AND ga4_sessions IS NOT NULL
              AND ga4_users IS NOT NULL
        ) AS ga4_complete

    FROM {MAR}
""").df()

availability_check

,all_rows,gsc_available,gsc_feature_fields_complete,ga4_available,ga4_complete
0,9841378,3611061,3611061,413966,413966


GSC data is available for 3,611,061 of the March rows, while GA4 data is available for only 413,966 rows. Since my clustering project focuses on search performance, I will use rows where gsc_data_available IS TRUE and avoid requiring GA4 data at this stage.

## 3.5 Feature frame


In [ ]:
# Build the five-feature frame for March 2026
feature_df = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS impressions,

        100.0 * SUM(gsc_clicks)
        / NULLIF(SUM(gsc_impressions), 0) AS ctr,

        SUM(gsc_sum_position)
        / NULLIF(SUM(gsc_impressions), 0) AS avg_position,

        COUNT(*) FILTER (
            WHERE gsc_impressions > 0
        ) AS days_with_impressions,

        COUNT(*) FILTER (
            WHERE gsc_clicks > 0
        ) AS days_with_clicks

    FROM {MAR}
    WHERE gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id

    HAVING SUM(gsc_impressions) > 0
""").df()

print(feature_df.shape[0], "pages,", feature_df.shape[1], "columns")
feature_df.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

176738 pages, 7 columns


,client_hash_id,content_hash_id,impressions,ctr,avg_position,days_with_impressions,days_with_clicks
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,1140.0,0.175439,4.450877,31,2
1,client_73cda7b4e4f265ea,content_05597932fe4da067,57.0,0.000000,2.298246,26,0
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,149.0,0.000000,5.637584,30,0
3,client_73cda7b4e4f265ea,content_05434271b257bb68,1421.0,0.422238,6.906404,31,5
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2770.0,0.577617,3.950542,31,6
5,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,48.0,0.000000,18.145833,21,0
6,client_73cda7b4e4f265ea,content_2662845f598544ef,150.0,0.666667,7.506667,30,1
7,client_73cda7b4e4f265ea,content_22610b0934f8825e,67.0,0.000000,12.000000,26,0
8,client_73cda7b4e4f265ea,content_712c365258cee05c,6048.0,0.380291,4.931878,31,16
9,client_73cda7b4e4f265ea,content_476c37c366920c1b,223.0,0.000000,61.538117,30,0


* **Impressions:** observed during March 2026.
* **CTR:** calculated from clicks and impressions observed during March.
* **Average position:** calculated from search-position data observed during March.
* **Days with impressions:** calculated from the days in March when the page appeared in search.
* **Days with clicks:** calculated from the days in March when the page received at least one click.

All five features use information observed or calculated from the March 2026 development window and do not use information from a future month.

## 3.5.5 Leakage trap

Because my main task is clustering, I do not normally have a target label. For this leakage demonstration, I create a temporary label showing whether a page has CTR above 0.

I then compare an honest model using only non-label-derived features with a second model that includes a column copied directly from the label. The second version is deliberately wrong and is only used to show how leakage can make a model look unrealistically good.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

leak_df = feature_df.copy()

# Temporary label: does the page have CTR above 0?
leak_df["has_ctr"] = (leak_df["ctr"] > 0).astype(int)

# Deliberately leaking feature: copy of the label
leak_df["label_copy"] = leak_df["has_ctr"]

honest_features = [
    "impressions",
    "avg_position",
    "days_with_impressions"
]

train, test = train_test_split(
    leak_df,
    test_size=0.2,
    random_state=42,
    stratify=leak_df["has_ctr"]
)

model = DecisionTreeClassifier(random_state=42)

# Honest model
model.fit(train[honest_features], train["has_ctr"])
honest_score = accuracy_score(
    test["has_ctr"],
    model.predict(test[honest_features])
)

# Leaky model
model.fit(train[honest_features + ["label_copy"]], train["has_ctr"])
leaky_score = accuracy_score(
    test["has_ctr"],
    model.predict(test[honest_features + ["label_copy"]])
)

print("Honest accuracy:", round(honest_score, 3))
print("Leaky accuracy:", round(leaky_score, 3))

# Remove the leaking feature
leak_df = leak_df.drop(columns="label_copy")


Honest accuracy: 0.785
Leaky accuracy: 1.0


The honest model reached an accuracy of **0.785**, while the model with the deliberately leaking `label_copy` feature reached **1.000**.

The perfect score is misleading because `label_copy` directly contains the answer the model is supposed to predict. After showing this effect, I removed the leaking feature and kept **0.785** as the honest score.


## 4. Data limits

One limitation is that I am using only March 2026 to build the clustering features. Patterns found in this month may reflect temporary or seasonal behaviour and may not represent how the same pages perform over a longer period.

I also only include rows where GSC data is available, so the resulting clusters will describe the GSC-available part of the warehouse rather than every content page.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.